In [8]:
import sys

import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON


In [9]:
print(sys.version)
print(pd.__version__)
print("SPARQLWrapper : OK")

3.14.10 | packaged by conda-forge | (main, Nov 27 2026, 03:43:02) [Clang 21.1.8 ]
3.1.1
SPARQLWrapper : OK


In [25]:
endpoint = SPARQLWrapper("https://query.wikidata.org/sparql")
query = """
SELECT ?person ?personLabel ?birthDate ?birthPlace ?birthPlaceLabel
WHERE {
    ?person wdt:P106 wd:Q2306091 .

    OPTIONAL { ?person wdt:P569 ?birthDate . }
    OPTIONAL { ?person wdt:P19 ?birthPlace . }

    SERVICE wikibase:label {
        bd:serviceParam wikibase:language "fr,en".
    }
}
LIMIT 500
"""

endpoint.setQuery(query)
endpoint.setReturnFormat(JSON)

results = endpoint.query().convert()

In [37]:
rows = [
    { key: value["value"] for key, value in row.items() }
    for row in results["results"]["bindings"]
]

df = pd.DataFrame(rows)
display(df)

print("Rows: ", len(df))
print("Unique persons: ", df["person"].nunique())
print("\nMissing values:")
print(df.isna().sum())


,person,birthDate,birthPlace,personLabel,birthPlaceLabel
0,http://www.wikidata.org/entity/Q1233,1896-09-25T00:00:00Z,http://www.wikidata.org/entity/Q3947233,Sandro Pertini,San Giovanni
1,http://www.wikidata.org/entity/Q5749,1809-01-15T00:00:00Z,http://www.wikidata.org/entity/Q37776,Pierre-Joseph Proudhon,Besançon
2,http://www.wikidata.org/entity/Q9061,1818-05-05T00:00:00Z,http://www.wikidata.org/entity/Q3138,Karl Marx,Trèves
3,http://www.wikidata.org/entity/Q9294,1332-06-04T00:00:00Z,http://www.wikidata.org/entity/Q3572,Ibn Khaldoun,Tunis
4,http://www.wikidata.org/entity/Q9387,1864-04-21T00:00:00Z,http://www.wikidata.org/entity/Q1729,Max Weber,Erfurt
...,...,...,...,...,...
495,http://www.wikidata.org/entity/Q210662,1858-12-31T00:00:00Z,http://www.wikidata.org/entity/Q37333,Zygmunt Balicki,Lublin
496,http://www.wikidata.org/entity/Q211095,1906-04-06T00:00:00Z,http://www.wikidata.org/entity/Q204150,George S. Wise,Pinsk
497,http://www.wikidata.org/entity/Q211771,1956-12-07T00:00:00Z,http://www.wikidata.org/entity/Q1780,Iveta Radičová,Bratislava
498,http://www.wikidata.org/entity/Q213652,1929-01-04T00:00:00Z,http://www.wikidata.org/entity/Q365,Amitai Etzioni,Cologne


Rows:  500
Unique persons:  474

Missing values:
person              0
birthDate           1
birthPlace         19
personLabel         0
birthPlaceLabel    19
dtype: int64


In [27]:
duplicates = df[df.duplicated(subset=['person'], keep=False)]

display(
    duplicates[
        ['personLabel', 'birthDate', 'birthPlaceLabel']
    ].head(20)
)

,personLabel,birthDate,birthPlaceLabel
11,Thomas Malthus,1766-02-13T00:00:00Z,Westcott
12,Thomas Malthus,1766-02-14T00:00:00Z,Westcott
13,Enid Mumford,1924-03-06T00:00:00Z,Merseyside
14,Enid Mumford,1924-03-06T00:00:00Z,Wallasey
29,H. G. Wells,1866-09-21T00:00:00Z,Londres
30,H. G. Wells,1866-09-21T00:00:00Z,Bromley
48,Hannah Arendt,1906-10-14T00:00:00Z,Hanovre
49,Hannah Arendt,1906-10-14T00:00:00Z,Lindener Marktplatz 2 (Hanovre)
50,Hannah Arendt,1906-10-14T00:00:00Z,Linden
82,Rita Süssmuth,1937-02-17T00:00:00Z,Allemagne


In [28]:
summary = df.groupby('person').agg(
    birthDates=('birthDate', 'nunique'),
    birthPlaces=('birthPlace', 'nunique')
)

print('Multiple birth dates: ', (summary['birthDates'] > 1).sum())
print('Multiple birth places: ', (summary['birthPlaces'] > 1).sum())

Multiple birth dates:  15
Multiple birth places:  9


In [39]:
def unique_values(series):
    return sorted(series.dropna().unique().tolist())

people = df.groupby(
    ["person", "personLabel"],
    as_index=False
).agg(
    birthDates=("birthDate", unique_values),
    birthPlaces=("birthPlace", unique_values)
)

print("Unique persons:", len(people))
display(df)

Unique persons: 474


,person,birthDate,birthPlace,personLabel,birthPlaceLabel
0,http://www.wikidata.org/entity/Q1233,1896-09-25T00:00:00Z,http://www.wikidata.org/entity/Q3947233,Sandro Pertini,San Giovanni
1,http://www.wikidata.org/entity/Q5749,1809-01-15T00:00:00Z,http://www.wikidata.org/entity/Q37776,Pierre-Joseph Proudhon,Besançon
2,http://www.wikidata.org/entity/Q9061,1818-05-05T00:00:00Z,http://www.wikidata.org/entity/Q3138,Karl Marx,Trèves
3,http://www.wikidata.org/entity/Q9294,1332-06-04T00:00:00Z,http://www.wikidata.org/entity/Q3572,Ibn Khaldoun,Tunis
4,http://www.wikidata.org/entity/Q9387,1864-04-21T00:00:00Z,http://www.wikidata.org/entity/Q1729,Max Weber,Erfurt
...,...,...,...,...,...
495,http://www.wikidata.org/entity/Q210662,1858-12-31T00:00:00Z,http://www.wikidata.org/entity/Q37333,Zygmunt Balicki,Lublin
496,http://www.wikidata.org/entity/Q211095,1906-04-06T00:00:00Z,http://www.wikidata.org/entity/Q204150,George S. Wise,Pinsk
497,http://www.wikidata.org/entity/Q211771,1956-12-07T00:00:00Z,http://www.wikidata.org/entity/Q1780,Iveta Radičová,Bratislava
498,http://www.wikidata.org/entity/Q213652,1929-01-04T00:00:00Z,http://www.wikidata.org/entity/Q365,Amitai Etzioni,Cologne
